# Quantitative Fund Dashboard
Run all cells in this notebook to visualize your portfolio data.

In [16]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
import anywidget

INITIAL_CAPITAL = 1_000_000_000
BASE_DIR = Path('../../inputs')

In [17]:
def load_historical_data(base_dir):
    nav_history = []
    holdings_history = []
    
    base_path = Path(base_dir)
    
    for dp_file in base_path.rglob('datapack.json'):
        try:
            with open(dp_file, 'r', encoding='utf-8') as f:
                dp = json.load(f)
            
            date = dp.get('trade_date')
            nav = dp.get('account', {}).get('nav_tradable', 0)
            cash = dp.get('account', {}).get('cash', 0)
            cash_ratio = dp.get('account', {}).get('cash_ratio_tradable', 0)

            nav_history.append({
                'Date': date,
                'NAV': nav,
                'Cash': cash,
                'Cash_Ratio': cash_ratio,
                'Profit_Pct': ((nav - INITIAL_CAPITAL) / INITIAL_CAPITAL) * 100,
                'Invested': nav - cash
            })

            for stock in dp.get('holdings', []):
                holdings_history.append({
                    'Date': date,
                    'Ticker': stock.get('code'),
                    'Weight': stock.get('tradable_weight', 0),
                    'Shares': stock.get('shares', 0),
                    'Close_Price': stock.get('close_price', 0)
                })
        except Exception as e:
            print(f'Error reading {dp_file}: {e}')

    if not nav_history:
        return pd.DataFrame(), pd.DataFrame()
        
    df_nav = pd.DataFrame(nav_history).sort_values('Date').reset_index(drop=True)
    df_holdings = pd.DataFrame(holdings_history).sort_values(['Date', 'Weight'], ascending=[True, False])
    return df_nav, df_holdings

df_nav, df_holdings = load_historical_data(BASE_DIR)

if df_nav.empty:
    print('⚠️ No datapack.json files found. Please ensure the inputs/ directory exists two levels up.')
else:
    print(f'✅ Successfully loaded {len(df_nav)} days of portfolio data.')

✅ Successfully loaded 6 days of portfolio data.


In [18]:
# 1. Cumulative Profit & Market Exposure
if not df_nav.empty:
    fig_profit = px.line(df_nav, x='Date', y='Profit_Pct', markers=True, 
                         title='Net Profit (%) Over Time',
                         labels={'Profit_Pct': 'Profit (%)'},
                         template='plotly_white')
    fig_profit.add_hline(y=0, line_dash='dash', line_color='red')
    fig_profit.show()

In [19]:
# 2. Capital Utilization (Invested vs Cash)
if not df_nav.empty:
    fig_cap = go.Figure()
    fig_cap.add_trace(go.Bar(x=df_nav['Date'], y=df_nav['Invested'], name='Invested Equity', marker_color='royalblue'))
    fig_cap.add_trace(go.Bar(x=df_nav['Date'], y=df_nav['Cash'], name='Cash Reserves', marker_color='lightgray'))
    fig_cap.update_layout(barmode='stack', title='Capital Utilization (Exposure Drift)', yaxis_title='NT$')
    fig_cap.show()

In [20]:
# 3. Interactive Daily Allocation Snapshot
if not df_nav.empty:
    dates = df_nav['Date'].tolist()
    date_dropdown = widgets.Dropdown(options=dates, value=dates[-1], description='Select Date:')
    
    # Using FigureWidget for in-place updates
    pie_fig = go.FigureWidget()
    pie_fig.add_pie(values=[], labels=[], hole=0.4)
    pie_fig.update_traces(textposition='inside', textinfo='percent+label')
    
    def update_pie_chart(change):
        selected_date = change['new']
        snapshot = df_holdings[df_holdings['Date'] == selected_date]
        cash_for_day = df_nav[df_nav['Date'] == selected_date]['Cash_Ratio'].values[0]
        
        snapshot_pie = pd.concat([
            snapshot[['Ticker', 'Weight']], 
            pd.DataFrame([{'Ticker': 'CASH', 'Weight': cash_for_day}])
        ])
        
        with pie_fig.batch_update():
            pie_fig.data[0].values = snapshot_pie['Weight']
            pie_fig.data[0].labels = snapshot_pie['Ticker']
            pie_fig.layout.title.text = f'Tradable Exposure ({selected_date})'
            
    date_dropdown.observe(update_pie_chart, names='value')
    display(widgets.VBox([date_dropdown, pie_fig]))
    update_pie_chart({'new': dates[-1]})

In [21]:
# 4. Historical Asset Evolution (Area Chart)
if not df_holdings.empty:
    df_pivot = df_holdings.pivot(index='Date', columns='Ticker', values='Weight').fillna(0)
    fig_area = px.area(df_pivot, title='Portfolio Transition Over Time', 
                       labels={'value': 'Weight (%)', 'variable': 'Ticker'})
    fig_area.show()

In [22]:
# 5. Individual Stock Performance Tracking
if not df_holdings.empty:
    from plotly.subplots import make_subplots
    import sys
    
    # Add the src folder to path to import universe.py
    if '..' not in sys.path:
        sys.path.append('..')
        
    try:
        from universe import load_universe
        universe = load_universe()
    except Exception:
        universe = None

    def get_dropdown_label(t):
        if universe:
            name = universe.names.get(t, '')
            theme = universe.theme_of(t)
            if name and theme:
                return f"{t} - {name} ({theme})"
            elif name:
                return f"{t} - {name}"
        return str(t)

    tickers = sorted(df_holdings['Ticker'].unique().tolist())
    dropdown_options = [(get_dropdown_label(t), t) for t in tickers]
    
    ticker_dropdown = widgets.Dropdown(options=dropdown_options, value=tickers[0], description='Ticker:')
    
    # Use FigureWidget to update in-place without duplicating outputs
    fig = go.FigureWidget(make_subplots(specs=[[{'secondary_y': True}]]))
    fig.add_trace(go.Scatter(x=[], y=[], name='Close Price', mode='lines+markers', line=dict(color='blue')), secondary_y=False)
    fig.add_trace(go.Bar(x=[], y=[], name='Portfolio Weight', opacity=0.3, marker_color='orange'), secondary_y=True)
    fig.update_layout(template='plotly_white', hovermode='x unified')
    fig.update_yaxes(title_text='<b>Close Price</b> (NT$)', secondary_y=False)
    fig.update_yaxes(title_text='<b>Portfolio Weight</b>', secondary_y=True, tickformat='.2%')

    def update_stock_chart(change):
        selected_ticker = change['new']
        stock_data = df_holdings[df_holdings['Ticker'] == selected_ticker].sort_values('Date')
        
        # Batch update to prevent flickering
        with fig.batch_update():
            fig.data[0].x = stock_data['Date']
            fig.data[0].y = stock_data['Close_Price']
            fig.data[1].x = stock_data['Date']
            fig.data[1].y = stock_data['Weight']
            
            # Find label
            label = selected_ticker
            for k, v in dropdown_options:
                if v == selected_ticker:
                    label = k
                    break
            fig.layout.title.text = f'Performance & Allocation Tracking: {label}'

    ticker_dropdown.observe(update_stock_chart, names='value')
    
    display(widgets.VBox([ticker_dropdown, fig]))
    
    update_stock_chart({'new': tickers[0]})